# Set Up pwd and auto updates

In [1]:
from pathlib import Path
import os
import subprocess
# Get the top-level directory of the current git repo
PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], text=True
    ).strip()
)

os.chdir(PROJECT_ROOT)


# Enable auto-reloading of custom modules
%load_ext autoreload
%autoreload 2

%pwd


'/Users/zanderholleran/Desktop/python_projects/mesa_lcc_model'

In [2]:

from season.configs import summarize_config
from collect_external_data.expected_counts import get_expected_counts
from collect_external_data.road_geom import get_road_geometry

from season.persons import SeasonPerson
from season.configs import ScheduleSpecs, SeasonConfig, DayParams, make_season_config, PopulationParams
from season.season_orchestrator import SeasonOrchestrator
from traffic.model.hybrid_collector import HybridCollectorConfig
from traffic.model.tolling import (
    TollConfig, 
    VolumeSignal, 
    FlowSignal, 
    PiecewiseLinearTransform, 
    StepTransform, 
    PITransform
)

import numpy as np
import pandas as pd
from pathlib import Path

from scipy.stats import norm, lognorm, skewnorm, truncnorm, uniform

import seaborn as sns



# Ensure Data Exists 

In [3]:
get_road_geometry()
get_expected_counts()


hw210_sl_and_curvs.parquet exists, skip re-processing
expected_counts_seconds.csv already exists, skip re-processing.


# Defining the Population Perams 

for small runs: 
- PopulationParams.population_size == make_season_config.max_persons & small n 
- 


In [4]:
config = make_season_config(
    # ── Identity ──
    season_id='speed_test6',
    run_description='',
    seed=33,

    # ── Simulation bounds ──
    n_days=3,
    max_steps=99999,
    start_hr=7,

    # ── Schedules (day-varying) ──
    traffic_percentile_schedule=ScheduleSpecs(mode='list', value=[85, 85, 85]),
    bus_interval_schedule=ScheduleSpecs(mode='static', value=30),
    crashes_schedule=ScheduleSpecs(mode='static', value=0),
    canyon_closures_schedule=None,

    # ── Population ──
    population_params=PopulationParams(
        population_size=1500,
        prior_car=22.0,
        prior_bus=40.0,
        time_decay_rate=0.1,
        prior_weight=1.0,
        uncertainty_multiplier=1.0,
    ),

    # ── Tolling ──
    toll=TollConfig(
        signal=VolumeSignal(),
        transform=PITransform(target=300, kp=0.5, ki=0.05, toll_min=0, toll_max=50),
        update_every_n_steps=60,
        rounding=0.10,
    ),
    bus_user_fee=0.0,
    bus_capacity=60,

    # ── Data collection ──
    collect_every_n=60,
    hybrid_collector_config=HybridCollectorConfig(
        max_steps=100000,
        tier1_enabled=True,
        tier2_enabled=False,
        tier3_enabled=False,
        tier4_enabled=False,
        tier1_interval=60,
        tier2_sample_interval=30,
        tier4_snapshot_interval=500,
        tier1_scalars=[
            'step', 'p_generate', 'current_toll', 'vehicle_count', 'active_cars', 'bus_riders_waiting',
            'active_buses', 'total_finished', 'bus_mode_share_recent',
        ],
        tier1_window_scalars=[
            'recent_travel_time_avg',
            'rolling_count_vehicles_generated',
            'rolling_count_persons_generated',
        ],
        tier1_histograms=['implicit_sl_delta'],
        tier2_max_samples=3000,
        tier2_max_agents_per_sample=150,
        tier4_snapshot_on_crash=True,
        tier4_max_snapshots=20,
        tier1_window_seconds=600,
    ),

)


summarize_config(config, high_only=False)

── Season Config ──
  Days: 3   Population: 1,500
  Traffic percentile: 85 (all days)
  Bus interval: 30 min   Bus capacity: 60
  Toll: PI → target 300, range $0–$50, rounded to $0.10

── Details ──
  Start hour: 7:00
  Value of time: lognorm (median ~$0.67/min)
  Priors: car=22.0, bus=40.0
  Belief params: decay=0.1, prior_wt=1.0, uncertainty_mult=1.0
  Data collection:
    • Tier 1 (scalars+histograms every 60 steps)


In [5]:
orchestrator = SeasonOrchestrator(season_config=config, store_data=True)
orchestrator.run_season()


Season outputs will be saved to: data/season_outputs/speed_test6


Simulating:  13%|█▎        | 12840/99999 [00:11<01:19, 1097.42step/s]


1500 people arrived stopping model.
{'RoadSegmentAgent': 402, 'TrafficPersonAgent': 1500}
Day:0: N:1500, Avg TT:34.7 min, Avg_cumtime_lost:9.0 min, Avg Cost (VOT standardized):$28.0, Avg Realized Cost:$31.6, bus_share:0.16, avg_tt_bus:79.6 min, avg_tt_car:26.2 min, avg_toll_car:$5.81, Total toll:$7315.50, 



Simulating:  14%|█▍        | 14432/99999 [00:13<01:20, 1068.42step/s]


1500 people arrived stopping model.
{'RoadSegmentAgent': 402, 'TrafficPersonAgent': 1500}
Day:1: N:1500, Avg TT:35.8 min, Avg_cumtime_lost:9.2 min, Avg Cost (VOT standardized):$29.8, Avg Realized Cost:$33.4, bus_share:0.16, avg_tt_bus:83.4 min, avg_tt_car:26.5 min, avg_toll_car:$7.06, Total toll:$8866.00, 



Simulating:  14%|█▍        | 14375/99999 [00:12<01:13, 1162.43step/s]

1500 people arrived stopping model.
{'RoadSegmentAgent': 402, 'TrafficPersonAgent': 1500}
Day:2: N:1500, Avg TT:38.7 min, Avg_cumtime_lost:9.3 min, Avg Cost (VOT standardized):$32.9, Avg Realized Cost:$37.4, bus_share:0.21, avg_tt_bus:85.5 min, avg_tt_car:26.5 min, avg_toll_car:$8.99, Total toll:$10706.30, 

Season Summary - Days Run: 3, Total Trips: 4500, 
Avg TT (all): 36.40, 
--- Cost Metrics --- 
     Marginal, Std VOT: $16.91, 
     Std VOT: $30.24, 
     All, Agent VOT: $34.10, 
     Bus, Agent VOT: $46.84, 
     Car, Agent VOT: $31.37, 
--- Tolling Metrics --- 
Avg Toll Cars: $7.25
     │███████████████████████───────────────┤
     Min: $0.00  Q1: $0.00  Med: $0.00  Q3: $15.60  Max: $25.50
Total Revenue: $26887.80


In [ ]:
# Read the day_0_model_ts parquet produced by the season run
parquet_path = PROJECT_ROOT / "data" / "season_outputs" / "speed_test2" / "day_2_model_ts.parquet"

if not parquet_path.exists():
    raise FileNotFoundError(f"Parquet file not found: {parquet_path}")

df_day = pd.read_parquet(parquet_path)

print(f"Loaded: {parquet_path}")
print("Shape:", df_day.shape)
print("\nColumn dtypes:")
print(df_day.dtypes)

# Show a quick sample
try:
    display(df_day.head(10))
except NameError:
    print(df_day.head(10))

In [ ]:
df_day

In [ ]:
orchestrator.last_model_run

In [ ]:
import cProfile
import pstats

# some stuff used for optimization

def main():
    # Example usage of SeasonOrchestrator with example_config
    orchestrator = SeasonOrchestrator(season_config=config, store_data=True)
    orchestrator.run_season()

if __name__ == "__main__":
    prof = cProfile.Profile()
    prof.enable()

    main()

    prof.disable()
    prof.dump_stats("prof.stats")

    p = pstats.Stats("prof.stats")
    p.strip_dirs().sort_stats("cumulative").print_stats(40)




# Example configs

In [ ]:
# ===================== Example Toll Configurations =====================
# Uncomment and use any of these in make_season_config(toll=..., bus_user_fee=...)

# 1. Static toll (fixed price)
# toll=TollConfig.static(car=10.0),

# 2. Volume-based piecewise linear (current config above)
# toll=TollConfig(
#     signal=VolumeSignal(),
#     transform=PiecewiseLinearTransform(threshold=100, slope=0.05, base=5.0),
#     update_every_n_steps=60,
#     rounding=0.25,
# ),

# 3. Flow-based piecewise linear (rolling average arrival rate)
# toll=TollConfig(
#     signal=FlowSignal(window_steps=300),  # 5-minute rolling window
#     transform=PiecewiseLinearTransform(threshold=1.0, slope=10.0, base=2.0),
#     update_every_n_steps=60,
#     rounding=0.25,
#     cap=25.0,  # max toll $25
# ),

# 4. Volume-based step toll (binary: $0 or $10)
# toll=TollConfig(
#     signal=VolumeSignal(),
#     transform=StepTransform(threshold=100, toll=10.0),
#     update_every_n_steps=60,
# ),

# 5. Volume-based PI controller (feedback-driven)
# toll=TollConfig(
#     signal=VolumeSignal(),
#     transform=PITransform(target=300, kp=0.5, ki=0.05, toll_min=0, toll_max=50),
#     update_every_n_steps=60,
#     rounding=0.10,
# ),